In [0]:
import sys
# Ensure parent directory is in path for module resolution
sys.path.append("/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline/")
from utils.metrics_query import MetricsQuery

mq = MetricsQuery(spark)

In [0]:
import sys
import os
from datetime import datetime
from pyspark.sql.functions import max as spark_max

# 1. Widgets & Setup
dbutils.widgets.text("customer_id", "")
dbutils.widgets.text("object_name", "")
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("bq_native_table", "v4c-bigquery.v4c_bigquery_dataset.event_raw")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
bq_native_table = dbutils.widgets.get("bq_native_table")
load_type = "incremental"

# --------------------------------------------------
# 2. Import Configuration dynamically
# --------------------------------------------------
# Root path where 'utils' folder resides
project_root = "/Workspace/Users/ayush.gunjal@hginsights.com/HGI-Lakehouse-Pipeline"
abs_project_root = os.path.abspath(project_root)

if abs_project_root not in sys.path:
    sys.path.append(abs_project_root)

from utils.config import DataLakeConfig

# Instantiate the config
config = DataLakeConfig(source_system, customer_id, object_name, load_type)

# Optimize shuffle for small clusters
spark.conf.set("spark.sql.shuffle.partitions", 1)

# Generate the timestamped path via Config Helper
now = datetime.now()
incremental_path = config.get_landing_zone_timestamped_path(source_system, customer_id, object_name, load_type, now)

print(f"🚀 Starting FAST NATIVE Incremental Load to: {incremental_path}")

gcp_project_id = bq_native_table.split(".")[0]

# 3. Securely Retrieve GCP Credentials
try:
    gcp_secret = dbutils.secrets.get(scope="gcp_auth", key="bq_key")
except Exception as e:
    raise Exception(f"❌ Failed to retrieve GCP credentials. Error: {str(e)}")

# 4. Get the last watermark
try:
    watermark_df = spark.sql(f"""
        SELECT last_processed_timestamp
        FROM ingestion_metadata.watermarks
        WHERE source_system='{source_system}'
        AND object_name='{object_name}'
    """)
    rows = watermark_df.collect()
    if len(rows) == 0:
        raise Exception("Watermark missing in ingestion_metadata.watermarks.")
    
    watermark_dt = rows[0][0] 
    # Format the Python datetime into BigQuery's string format (ISO 8601)
    watermark_str = watermark_dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
    
except Exception as e:
    raise Exception(f"❌ Watermark Error: Please run historical load first. Error: {str(e)}")

print(f"📍 Last Watermark: {watermark_str}")

# 5. Query BigQuery for the NEW max timestamp
new_ts_df = spark.read \
    .format("bigquery") \
    .option("credentials", gcp_secret) \
    .option("parentProject", gcp_project_id) \
    .option("project", gcp_project_id) \
    .option("filter", f"event_timestamp > '{watermark_str}'") \
    .load(bq_native_table) \
    .select(spark_max("event_timestamp").alias("max_ts"))

new_ts_str = new_ts_df.first()[0]

if new_ts_str is None:
    print("✅ No new records found in BigQuery. Exiting gracefully.")
    dbutils.notebook.exit("0")

print(f"🎯 New records found up to: {new_ts_str}")

# 6. Pull the data using Native Connector
df_incremental = spark.read \
    .format("bigquery") \
    .option("credentials", gcp_secret) \
    .option("parentProject", gcp_project_id) \
    .option("project", gcp_project_id) \
    .option("filter", f"event_timestamp > '{watermark_str}' AND event_timestamp <= '{new_ts_str}'") \
    .load(bq_native_table)

try:
    # Write to S3 using config path
    df_incremental.repartition(2).write \
        .mode("append") \
        .format("parquet") \
        .option("compression", "snappy") \
        .save(incremental_path)
    print("💾 Write to S3 completed successfully.")
    
except Exception as e:
    print(f"❌ Failed during write operation: {str(e)}")
    raise e

# 7. Update Watermark Table
print("Updating watermark table with lookback window...")

spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW new_watermarks AS
    SELECT source_system, object_name, 
           CASE 
               WHEN source_system = '{source_system}' AND object_name = '{object_name}' 
               THEN CAST('{new_ts_str}' AS TIMESTAMP) - INTERVAL 1 MINUTE
               ELSE last_processed_timestamp 
           END as last_processed_timestamp
    FROM ingestion_metadata.watermarks
""")

spark.sql("INSERT OVERWRITE TABLE ingestion_metadata.watermarks SELECT * FROM new_watermarks")

print(f"✅ Watermark updated successfully (minus 1 minute).")

In [0]:
# Cell 8 — use actual resolved table name
# No table is created, so skip row count and just save stats with rows_processed=None
mq.save_stats(days=30, rows_processed=None)
print(f":white_check_mark: Metrics saved | rows_processed: None for run_id: {mq._run_id}")
